In [124]:
# basic imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [125]:
df_1 = pd.read_csv("raw/scraped_steam_games.csv")
df_1.head()


,Name,Release Date,Price,Discount,Developer,Publisher,Total Reviews,Positive Reviews,Negative Reviews,Review Sentiment,Genre(s)
0,Counter-Strike 2,"Aug 21, 2012",Free To Play,No Discount,Valve,Valve,"9,769,323","8,403,080","1,366,243",Mostly Positive,"['Action', 'Free To Play']"
1,Palworld,"Jul 9, 2026",$29.99,No Discount,Pocketpair,Pocketpair,"467,020","441,183","25,837",Very Positive,"['Action', 'Adventure', 'Indie', 'RPG']"
2,Baldur's Gate 3,"Aug 3, 2023",$59.99,$41.99,Larian Studios,Larian Studios,"851,003","823,690","27,313",Very Positive,"['Adventure', 'RPG', 'Strategy']"
3,Marvel Rivals,"Dec 5, 2024",Free To Play,No Discount,NetEase Games,NetEase Games,"405,604","307,886","97,718",Mixed,"['Action', 'Free To Play']"
4,Warframe,"Mar 25, 2013",Free To Play,No Discount,Digital Extremes,Digital Extremes,"678,984","593,734","85,250",Very Positive,"['Action', 'RPG', 'Free To Play']"


In [126]:
df_2 = pd.read_csv("raw/scraped_steam_games_2.csv")
df_2.head()

,Name,Release Date,Price,Discount,Developer,Publisher,Total Reviews,Positive Reviews,Negative Reviews,Review Sentiment,Genre(s)
0,Minecraft Dungeons II,"Sep 29, 2026",$29.99,No Discount,Mojang Studios,Xbox Game Studios,0,0,0,"['Action', 'Adventure', 'RPG']",NaN
1,Out of Ore,"Oct 31, 2022",$24.99,No Discount,North Modding Company AB,North Modding Company AB,"4,087","3,504",583,Mostly Positive,"['Adventure', 'Casual', 'Simulation', 'Strategy']"
2,House Flipper 2 - Scooby-Doo DLC,"Oct 23, 2025",$14.99,No Discount,Frozen District,Frozen District,196,172,24,Very Positive,"['Casual', 'Indie', 'Simulation']"
3,Foxhole,"Sep 28, 2022",$29.99,No Discount,Siege Camp,Siege Camp,"52,720","41,940","10,780",Mixed,"['Action', 'Indie', 'Massively Multiplayer', '..."
4,DRAGON QUEST I & II HD-2D Remake,"Oct 30, 2025",$59.99,No Discount,Square Enix,Square Enix,"2,005","1,712",293,Very Positive,['RPG']


In [127]:
#concat the two dataframes together
df = pd.concat([df_1, df_2])
print(df.shape)

(1995, 11)


# Data Cleaning

In [128]:
#go feature by feature
#date parse
df["Release Date"] = pd.to_datetime(df["Release Date"], format = "mixed")
print(df["Release Date"])

0     2012-08-21
1     2026-07-09
2     2023-08-03
3     2024-12-05
4     2013-03-25
         ...    
994   2026-06-12
995   2023-09-20
996   2025-11-20
997   2019-12-10
998   2024-01-30
Name: Release Date, Length: 1995, dtype: datetime64[us]


In [129]:
#price
df["Price"] = df["Price"].replace("Free To Play", "0")
df["Price"] = df["Price"].replace(to_replace = r"(?i).*Free.*", value = "0", regex = True)
df["Price"] = df["Price"].replace("No Price", "0")
df["Price"] = df["Price"].str.replace("/ month", "", regex=False)
df["Price"] = df["Price"].replace(to_replace = r"(?i).*demo.*", value = "0", regex = True)
df["Price"] = df["Price"].str.strip("$")
df["Price"] = df["Price"].astype("float")


In [130]:
#discount
df["Discount"] = df["Discount"].replace("No Discount", "0")
df["Discount"] = df["Discount"].str.strip("$")
df["Discount"] = df["Discount"].astype("float")

In [131]:
df["Total Reviews"] = df["Total Reviews"].str.replace(",", "")
df["Total Reviews"] = df["Total Reviews"].astype("int64")

df["Positive Reviews"] = df["Positive Reviews"].str.replace(",", "")
df["Positive Reviews"] = df["Positive Reviews"].astype("int64")

df["Negative Reviews"] = df["Negative Reviews"].str.replace(",", "")
df["Negative Reviews"] = df["Negative Reviews"].astype("int64")

print(df.dtypes)


Name                           str
Release Date        datetime64[us]
Price                      float64
Discount                   float64
Developer                      str
Publisher                      str
Total Reviews                int64
Positive Reviews             int64
Negative Reviews             int64
Review Sentiment               str
Genre(s)                       str
dtype: object


In [132]:
#Separate the genres
import ast
df["Genre(s)"] = df["Genre(s)"].fillna("['None', 'None', 'None', 'None']")
df["Genre(s)"] = df["Genre(s)"].apply(ast.literal_eval)
df["Genre1"] = df["Genre(s)"].str[0]
df["Genre2"] = df["Genre(s)"].str[1]
df["Genre3"] = df["Genre(s)"].str[2]

df.head()


,Name,Release Date,Price,Discount,Developer,Publisher,Total Reviews,Positive Reviews,Negative Reviews,Review Sentiment,Genre(s),Genre1,Genre2,Genre3
0,Counter-Strike 2,2012-08-21,0.00,0.00,Valve,Valve,9769323,8403080,1366243,Mostly Positive,"[Action, Free To Play]",Action,Free To Play,NaN
1,Palworld,2026-07-09,29.99,0.00,Pocketpair,Pocketpair,467020,441183,25837,Very Positive,"[Action, Adventure, Indie, RPG]",Action,Adventure,Indie
2,Baldur's Gate 3,2023-08-03,59.99,41.99,Larian Studios,Larian Studios,851003,823690,27313,Very Positive,"[Adventure, RPG, Strategy]",Adventure,RPG,Strategy
3,Marvel Rivals,2024-12-05,0.00,0.00,NetEase Games,NetEase Games,405604,307886,97718,Mixed,"[Action, Free To Play]",Action,Free To Play,NaN
4,Warframe,2013-03-25,0.00,0.00,Digital Extremes,Digital Extremes,678984,593734,85250,Very Positive,"[Action, RPG, Free To Play]",Action,RPG,Free To Play


In [133]:
#Data Cleaning is complete

# New Feature Creation

In [134]:
#Year Released
df["Year Released"] = df["Release Date"].dt.year

In [138]:
#Discount Percentage
df["Discount %"] = round((df["Price"]-df["Discount"])/df["Price"] * 100)
df["Discount %"] = df["Discount %"].replace(100.0, None)

In [140]:
df["Positive Reviews %"] = df["Positive Reviews"]/df["Total Reviews"] * 100
df["Negative Reviews %"] = df["Negative Reviews"]/df["Total Reviews"] * 100
df.head(15)

,Name,Release Date,Price,Discount,Developer,Publisher,Total Reviews,Positive Reviews,Negative Reviews,Review Sentiment,Genre(s),Genre1,Genre2,Genre3,Year Released,Discount %,Positive Reviews %,Negative Reviews %
0,Counter-Strike 2,2012-08-21,0.00,0.00,Valve,Valve,9769323,8403080,1366243,Mostly Positive,"[Action, Free To Play]",Action,Free To Play,NaN,2012.0,NaN,86.014967,13.985033
1,Palworld,2026-07-09,29.99,0.00,Pocketpair,Pocketpair,467020,441183,25837,Very Positive,"[Action, Adventure, Indie, RPG]",Action,Adventure,Indie,2026.0,None,94.467689,5.532311
2,Baldur's Gate 3,2023-08-03,59.99,41.99,Larian Studios,Larian Studios,851003,823690,27313,Very Positive,"[Adventure, RPG, Strategy]",Adventure,RPG,Strategy,2023.0,30.0,96.790493,3.209507
3,Marvel Rivals,2024-12-05,0.00,0.00,NetEase Games,NetEase Games,405604,307886,97718,Mixed,"[Action, Free To Play]",Action,Free To Play,NaN,2024.0,NaN,75.908029,24.091971
4,Warframe,2013-03-25,0.00,0.00,Digital Extremes,Digital Extremes,678984,593734,85250,Very Positive,"[Action, RPG, Free To Play]",Action,RPG,Free To Play,2013.0,NaN,87.444476,12.555524
5,Marvel's Spider-Man 2,2025-01-30,59.99,40.19,Insomniac Games,PlayStation Publishing LLC,42196,34923,7273,Very Positive,"[Action, Adventure]",Action,Adventure,NaN,2025.0,33.0,82.763769,17.236231
6,Mistfall Hunter,2026-07-29,24.99,22.49,Bellring Games,Skystone Games,6358,4208,2150,Mixed,"[Action, Adventure]",Action,Adventure,NaN,2026.0,10.0,66.184335,33.815665
7,Cyberpunk 2077,2020-12-09,59.99,17.99,CD PROJEKT RED,CD PROJEKT RED,974300,840706,133594,Overwhelmingly Positive,[RPG],RPG,NaN,NaN,2020.0,70.0,86.288207,13.711793
8,Assassin's Creed Black Flag Resynced,2026-07-09,59.99,0.00,Ubisoft Singapore,Ubisoft,25840,20498,5342,Very Positive,"[Action, Adventure]",Action,Adventure,NaN,2026.0,None,79.326625,20.673375
9,Apex Legends™,2020-11-04,0.00,0.00,Respawn,Electronic Arts,1061120,718572,342548,Mixed,"[Action, Adventure, Free To Play]",Action,Adventure,Free To Play,2020.0,NaN,67.718260,32.281740
